In [2]:
from pathlib import Path

import json
import joblib
import numpy as np
import pandas as pd

from tensorflow.keras.models import load_model


# ============================================================
# PROJECT ROOT
# ============================================================

project_root = Path.cwd().resolve()

while (
    project_root.name != "football-prediction"
    and project_root.parent != project_root
):
    project_root = project_root.parent

if project_root.name != "football-prediction":
    raise FileNotFoundError(
        "Could not locate the football-prediction project root."
    )


# ============================================================
# DATA PATHS
# ============================================================

temporal_data_path = (
    project_root
    / "data"
    / "processed"
    / "temporal_market_value_dataset.csv"
)

players_path = (
    project_root
    / "data"
    / "raw"
    / "players.csv"
)


# ============================================================
# LSTM DEPLOYMENT PATHS
# ============================================================

model_dir = (
    project_root
    / "models"
    / "experiment_2"
)

lstm_model_path = (
    model_dir
    / "lstm_future_market_value.keras"
)

feature_scaler_path = (
    model_dir
    / "lstm_feature_scaler.joblib"
)

target_scaler_path = (
    model_dir
    / "lstm_target_scaler.joblib"
)

metadata_path = (
    model_dir
    / "lstm_future_market_value_metadata.json"
)


# ============================================================
# PATH CHECK
# ============================================================

print("LSTM DEPLOYMENT PATH CHECK")
print("=" * 70)

print(f"Project root    : {project_root}")

print("\nData files:")
print(
    f"Temporal dataset: "
    f"{temporal_data_path.exists()}"
)

print(
    f"Players dataset : "
    f"{players_path.exists()}"
)

print("\nDeployment files:")
print(
    f"LSTM model      : "
    f"{lstm_model_path.exists()}"
)

print(
    f"Feature scaler  : "
    f"{feature_scaler_path.exists()}"
)

print(
    f"Target scaler   : "
    f"{target_scaler_path.exists()}"
)

print(
    f"Metadata        : "
    f"{metadata_path.exists()}"
)

LSTM DEPLOYMENT PATH CHECK
Project root    : C:\Users\ngtht\football-prediction

Data files:
Temporal dataset: True
Players dataset : True

Deployment files:
LSTM model      : True
Feature scaler  : True
Target scaler   : True
Metadata        : True


In [3]:
# ============================================================
# LOAD SAVED DEPLOYMENT ARTIFACTS
# ============================================================

lstm_model = load_model(
    lstm_model_path
)

feature_scaler = joblib.load(
    feature_scaler_path
)

target_scaler = joblib.load(
    target_scaler_path
)

with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as file:
    metadata = json.load(file)


print("LSTM DEPLOYMENT ARTIFACTS")
print("=" * 70)

print(
    f"Model type       : "
    f"{type(lstm_model)}"
)

print(
    f"Feature scaler   : "
    f"{type(feature_scaler)}"
)

print(
    f"Target scaler    : "
    f"{type(target_scaler)}"
)

print("\nMODEL CONFIGURATION")
print("-" * 70)

print(
    f"Model            : "
    f"{metadata['model_name']}"
)

print(
    f"Sequence length  : "
    f"{metadata['sequence_length']}"
)

print(
    f"Number features  : "
    f"{metadata['number_of_features']}"
)

print(
    f"Maximum gap      : "
    f"{metadata['maximum_allowed_gap_days']} days"
)

print(
    f"Target           : "
    f"{metadata['target']}"
)

print(
    f"Forecast horizon : "
    f"{metadata['forecast_horizon']}"
)

print("\nFeatures:")

for i, feature in enumerate(
    metadata["features"],
    start=1
):
    print(f"{i}. {feature}")

LSTM DEPLOYMENT ARTIFACTS
Model type       : <class 'keras.src.models.sequential.Sequential'>
Feature scaler   : <class 'sklearn.preprocessing._data.StandardScaler'>
Target scaler    : <class 'sklearn.preprocessing._data.StandardScaler'>

MODEL CONFIGURATION
----------------------------------------------------------------------
Model            : LSTM
Sequence length  : 3
Number features  : 4
Maximum gap      : 240 days
Target           : future_market_value
Forecast horizon : Approximately 6 months

Features:
1. current_market_value
2. age_at_snapshot
3. minutes_last_6m
4. goals_last_6m


In [4]:
# ============================================================
# LOAD DEPLOYMENT DATA
# ============================================================

temporal_df = pd.read_csv(
    temporal_data_path
)

players_df = pd.read_csv(
    players_path
)

# Convert dates
temporal_df["snapshot_date"] = pd.to_datetime(
    temporal_df["snapshot_date"],
    errors="coerce"
)

if "date_of_birth" in players_df.columns:
    players_df["date_of_birth"] = pd.to_datetime(
        players_df["date_of_birth"],
        errors="coerce"
    )


print("DEPLOYMENT DATA")
print("=" * 70)

print(
    f"Temporal dataset shape : "
    f"{temporal_df.shape}"
)

print(
    f"Unique players         : "
    f"{temporal_df['player_id'].nunique():,}"
)

print(
    f"Snapshot range         : "
    f"{temporal_df['snapshot_date'].min().date()} "
    f"to "
    f"{temporal_df['snapshot_date'].max().date()}"
)

print(
    f"Players dataset shape  : "
    f"{players_df.shape}"
)

DEPLOYMENT DATA
Temporal dataset shape : (213905, 26)
Unique players         : 24,275
Snapshot range         : 2012-07-06 to 2025-12-22
Players dataset shape  : (47669, 26)


In [5]:
# ============================================================
# VALIDATE DEPLOYMENT REQUIREMENTS
# ============================================================

model_features = metadata["features"]

required_columns = [
    "player_id",
    "snapshot_date"
] + model_features

missing_columns = [
    column
    for column in required_columns
    if column not in temporal_df.columns
]

print("DEPLOYMENT REQUIREMENT CHECK")
print("=" * 70)

print(
    f"Required sequence length : "
    f"{metadata['sequence_length']}"
)

print(
    f"Required features        : "
    f"{len(model_features)}"
)

print(
    f"Maximum allowed gap      : "
    f"{metadata['maximum_allowed_gap_days']} days"
)

print(
    f"Missing columns          : "
    f"{missing_columns}"
)

if missing_columns:
    raise ValueError(
        "Temporal dataset is missing required "
        f"deployment columns: {missing_columns}"
    )

print("\nAll required deployment columns are available.")

DEPLOYMENT REQUIREMENT CHECK
Required sequence length : 3
Required features        : 4
Maximum allowed gap      : 240 days
Missing columns          : []

All required deployment columns are available.


In [6]:
# ============================================================
# IDENTIFY PLAYERS ELIGIBLE FOR LSTM DEPLOYMENT
# ============================================================

sequence_length = metadata["sequence_length"]
max_gap_days = metadata["maximum_allowed_gap_days"]

# Sort temporal observations chronologically
deployment_df = (
    temporal_df
    .sort_values(
        ["player_id", "snapshot_date"]
    )
    .reset_index(drop=True)
)

eligible_players = []

for player_id, player_data in deployment_df.groupby("player_id"):

    player_data = (
        player_data
        .sort_values("snapshot_date")
        .reset_index(drop=True)
    )

    # Need at least 3 historical observations
    if len(player_data) < sequence_length:
        continue

    # Take latest 3 observations
    latest_sequence = player_data.tail(
        sequence_length
    ).copy()

    dates = (
        latest_sequence["snapshot_date"]
        .reset_index(drop=True)
    )

    gaps = dates.diff().dt.days.dropna()

    # Both gaps must satisfy deployment constraint
    if (gaps <= max_gap_days).all():

        # Also require complete model features
        if not latest_sequence[model_features].isna().any().any():

            eligible_players.append({
                "player_id": player_id,
                "snapshot_t_minus_2": dates.iloc[0],
                "snapshot_t_minus_1": dates.iloc[1],
                "snapshot_t": dates.iloc[2],
                "gap_1_days": int(gaps.iloc[0]),
                "gap_2_days": int(gaps.iloc[1])
            })


eligible_players_df = pd.DataFrame(
    eligible_players
)

print("LSTM DEPLOYMENT ELIGIBILITY")
print("=" * 70)

print(
    f"Total players          : "
    f"{deployment_df['player_id'].nunique():,}"
)

print(
    f"Eligible players       : "
    f"{len(eligible_players_df):,}"
)

print(
    f"Eligibility percentage : "
    f"{len(eligible_players_df) / deployment_df['player_id'].nunique() * 100:.2f}%"
)

display(
    eligible_players_df.head(10)
)

LSTM DEPLOYMENT ELIGIBILITY
Total players          : 24,275
Eligible players       : 15,785
Eligibility percentage : 65.03%


,player_id,snapshot_t_minus_2,snapshot_t_minus_1,snapshot_t,gap_1_days,gap_2_days
0,10,2014-07-07,2015-01-07,2015-07-01,184,175
1,65,2014-01-15,2014-08-07,2015-01-14,204,160
2,80,2017-02-07,2017-06-19,2017-12-28,132,192
3,215,2021-12-09,2022-06-14,2022-11-30,187,169
4,258,2013-08-26,2014-01-02,2014-07-27,129,206
5,264,2016-09-30,2017-02-23,2017-07-31,146,158
6,277,2014-07-28,2015-01-13,2015-07-01,169,169
7,332,2016-02-11,2016-07-22,2017-01-23,162,185
8,488,2015-07-01,2016-02-15,2016-08-01,229,168
9,532,2018-06-05,2018-12-18,2019-06-05,196,169


In [7]:
# ============================================================
# ADD PLAYER PROFILE INFORMATION
# ============================================================

profile_columns = [
    "player_id",
    "name",
    "position",
    "sub_position",
    "current_club_name",
    "country_of_citizenship",
    "image_url"
]

available_profile_columns = [
    column
    for column in profile_columns
    if column in players_df.columns
]

eligible_players_df = eligible_players_df.merge(
    players_df[available_profile_columns],
    on="player_id",
    how="left"
)

print("ELIGIBLE PLAYER PROFILES")
print("=" * 70)

print(
    f"Eligible players : "
    f"{len(eligible_players_df):,}"
)

if "name" in eligible_players_df.columns:
    print(
        f"Missing names    : "
        f"{eligible_players_df['name'].isna().sum():,}"
    )

display(
    eligible_players_df.head(10)
)

ELIGIBLE PLAYER PROFILES
Eligible players : 15,785
Missing names    : 0


,player_id,snapshot_t_minus_2,snapshot_t_minus_1,snapshot_t,gap_1_days,gap_2_days,name,position,sub_position,current_club_name,country_of_citizenship,image_url
0,10,2014-07-07,2015-01-07,2015-07-01,184,175,Miroslav Klose,Attack,Centre-Forward,Società Sportiva Lazio S.p.A.,Germany,https://img.a.transfermarkt.technology/portrai...
1,65,2014-01-15,2014-08-07,2015-01-14,204,160,Dimitar Berbatov,Attack,Centre-Forward,Panthessalonikios Athlitikos Omilos Konstantin...,Bulgaria,https://img.a.transfermarkt.technology/portrai...
2,80,2017-02-07,2017-06-19,2017-12-28,132,192,Tom Starke,Goalkeeper,Goalkeeper,FC Bayern München,Germany,https://img.a.transfermarkt.technology/portrai...
3,215,2021-12-09,2022-06-14,2022-11-30,187,169,Roque Santa Cruz,Attack,Centre-Forward,Málaga CF,Paraguay,https://img.a.transfermarkt.technology/portrai...
4,258,2013-08-26,2014-01-02,2014-07-27,129,206,Gerald Asamoah,Attack,Centre-Forward,FC Schalke 04,Germany,https://img.a.transfermarkt.technology/portrai...
5,264,2016-09-30,2017-02-23,2017-07-31,146,158,Tamás Hajnal,Midfield,Attacking Midfield,Verein für Bewegungsspiele Stuttgart 1893,Hungary,https://img.a.transfermarkt.technology/portrai...
6,277,2014-07-28,2015-01-13,2015-07-01,169,169,Sérgio da Silva Pinto,Midfield,Central Midfield,Levante Unión Deportiva S.A.D.,Germany,https://img.a.transfermarkt.technology/portrai...
7,332,2016-02-11,2016-07-22,2017-01-23,162,185,Benjamin Köhler,Midfield,Left Midfield,Eintracht Frankfurt Fußball AG,Germany,https://img.a.transfermarkt.technology/portrai...
8,488,2015-07-01,2016-02-15,2016-08-01,229,168,Gerhard Tremmel,Goalkeeper,Goalkeeper,Swansea City,Germany,https://img.a.transfermarkt.technology/portrai...
9,532,2018-06-05,2018-12-18,2019-06-05,196,169,Claudio Pizarro,Attack,Centre-Forward,Sportverein Werder Bremen von 1899,Peru,https://img.a.transfermarkt.technology/portrai...


In [8]:
# ============================================================
# DEPLOYMENT EXAMPLE: COLE PALMER
# ============================================================

player_search = eligible_players_df[
    eligible_players_df["name"]
    .str.contains(
        "Cole Palmer",
        case=False,
        na=False
    )
]

print("PLAYER SEARCH RESULT")
print("=" * 70)

display(player_search)

PLAYER SEARCH RESULT


,player_id,snapshot_t_minus_2,snapshot_t_minus_1,snapshot_t,gap_1_days,gap_2_days,name,position,sub_position,current_club_name,country_of_citizenship,image_url
13226,568177,2024-05-27,2024-12-16,2025-05-30,203,165,Cole Palmer,Midfield,Attacking Midfield,Chelsea Football Club,England,https://img.a.transfermarkt.technology/portrai...


In [9]:
# ============================================================
# EXTRACT PLAYER'S LATEST LSTM SEQUENCE
# ============================================================

selected_player_id = int(
    player_search.iloc[0]["player_id"]
)

selected_player_name = (
    player_search.iloc[0]["name"]
)

player_history = (
    deployment_df[
        deployment_df["player_id"]
        == selected_player_id
    ]
    .sort_values("snapshot_date")
    .reset_index(drop=True)
)

player_sequence = (
    player_history
    .tail(sequence_length)
    .copy()
    .reset_index(drop=True)
)

player_sequence.index = [
    "t-2",
    "t-1",
    "t"
]

print("PLAYER LSTM SEQUENCE")
print("=" * 70)

print(
    f"Player : "
    f"{selected_player_name}"
)

print(
    f"Player ID : "
    f"{selected_player_id}"
)

display(
    player_sequence[
        ["snapshot_date"] + model_features
    ]
)

PLAYER LSTM SEQUENCE
Player : Cole Palmer
Player ID : 568177


,snapshot_date,current_market_value,age_at_snapshot,minutes_last_6m,goals_last_6m
t-2,2024-05-27,80000000,22.058864,2781.0,21.0
t-1,2024-12-16,130000000,22.614648,1442.0,11.0
t,2025-05-30,120000000,23.066393,2243.0,8.0


In [10]:
# ============================================================
# VALIDATE SELECTED DEPLOYMENT SEQUENCE
# ============================================================

sequence_dates = (
    player_sequence["snapshot_date"]
    .reset_index(drop=True)
)

sequence_gaps = (
    sequence_dates
    .diff()
    .dt.days
    .dropna()
    .to_numpy()
)

sequence_has_missing = (
    player_sequence[model_features]
    .isna()
    .any()
    .any()
)

gap_check = (
    sequence_gaps <= max_gap_days
).all()

print("SELECTED SEQUENCE VALIDATION")
print("=" * 70)

print(
    f"Sequence length     : "
    f"{len(player_sequence)}"
)

print(
    f"Expected length     : "
    f"{sequence_length}"
)

print(
    f"Gap t-2 → t-1      : "
    f"{sequence_gaps[0]} days"
)

print(
    f"Gap t-1 → t        : "
    f"{sequence_gaps[1]} days"
)

print(
    f"Maximum allowed gap: "
    f"{max_gap_days} days"
)

print(
    f"Gap check passed    : "
    f"{gap_check}"
)

print(
    f"Missing features    : "
    f"{sequence_has_missing}"
)

deployment_ready = (
    len(player_sequence) == sequence_length
    and gap_check
    and not sequence_has_missing
)

print(
    f"\nDeployment ready    : "
    f"{deployment_ready}"
)

SELECTED SEQUENCE VALIDATION
Sequence length     : 3
Expected length     : 3
Gap t-2 → t-1      : 203.0 days
Gap t-1 → t        : 165.0 days
Maximum allowed gap: 240 days
Gap check passed    : True
Missing features    : False

Deployment ready    : True


In [11]:
# ============================================================
# PREPARE LSTM DEPLOYMENT INPUT
# ============================================================

if not deployment_ready:
    raise ValueError(
        "Selected player sequence is not deployment-ready."
    )

# Extract raw feature matrix
# Shape should be: (3 timesteps, 4 features)
sequence_raw = (
    player_sequence[model_features]
    .to_numpy(dtype=np.float64)
)

print("RAW DEPLOYMENT SEQUENCE")
print("=" * 70)

print(
    f"Raw sequence shape : "
    f"{sequence_raw.shape}"
)

print("\nFeature order:")

for i, feature in enumerate(
    model_features,
    start=1
):
    print(f"{i}. {feature}")

print("\nRaw input:")
print(sequence_raw)

RAW DEPLOYMENT SEQUENCE
Raw sequence shape : (3, 4)

Feature order:
1. current_market_value
2. age_at_snapshot
3. minutes_last_6m
4. goals_last_6m

Raw input:
[[8.00000000e+07 2.20588638e+01 2.78100000e+03 2.10000000e+01]
 [1.30000000e+08 2.26146475e+01 1.44200000e+03 1.10000000e+01]
 [1.20000000e+08 2.30663929e+01 2.24300000e+03 8.00000000e+00]]


In [12]:
# ============================================================
# SCALE DEPLOYMENT SEQUENCE
# ============================================================

sequence_scaled_2d = (
    feature_scaler.transform(
        sequence_raw
    )
)

# Add batch dimension required by LSTM
sequence_scaled_3d = (
    sequence_scaled_2d.reshape(
        1,
        sequence_length,
        len(model_features)
    )
)

print("SCALED LSTM INPUT")
print("=" * 70)

print(
    f"Before scaling shape : "
    f"{sequence_raw.shape}"
)

print(
    f"After scaling shape  : "
    f"{sequence_scaled_2d.shape}"
)

print(
    f"LSTM input shape     : "
    f"{sequence_scaled_3d.shape}"
)

print("\nScaled sequence:")
print(sequence_scaled_2d)

print("\nNaN check:")
print(
    np.isnan(sequence_scaled_3d).sum()
)

print("\nInf check:")
print(
    np.isinf(sequence_scaled_3d).sum()
)

SCALED LSTM INPUT
Before scaling shape : (3, 4)
After scaling shape  : (3, 4)
LSTM input shape     : (1, 3, 4)

Scaled sequence:
[[ 8.19302865 -1.11681696  3.32919925  9.65519946]
 [13.58815185 -0.98582201  1.30496773  4.85990721]
 [12.50912721 -0.87934828  2.51587844  3.42131953]]

NaN check:
0

Inf check:
0


In [13]:
# ============================================================
# LSTM DEPLOYMENT PREDICTION
# ============================================================

# Predict standardised target
prediction_scaled = lstm_model.predict(
    sequence_scaled_3d,
    verbose=0
)

# Convert prediction back to EUR
prediction_eur = (
    target_scaler
    .inverse_transform(
        prediction_scaled.reshape(-1, 1)
    )
    .reshape(-1)[0]
)

# Market value cannot be negative
prediction_eur = max(
    0.0,
    float(prediction_eur)
)

# Current market value = latest timestep t
current_market_value = float(
    player_sequence.loc[
        "t",
        "current_market_value"
    ]
)

absolute_change = (
    prediction_eur
    - current_market_value
)

percentage_change = (
    absolute_change
    / current_market_value
    * 100
)

latest_snapshot_date = (
    player_sequence.loc[
        "t",
        "snapshot_date"
    ]
)

forecast_date = (
    latest_snapshot_date
    + pd.DateOffset(months=6)
)


print("LSTM DEPLOYMENT PREDICTION")
print("=" * 70)

print(
    f"Player                 : "
    f"{selected_player_name}"
)

print(
    f"Latest snapshot        : "
    f"{latest_snapshot_date.date()}"
)

print(
    f"Approx. forecast date  : "
    f"{forecast_date.date()}"
)

print(
    f"Current market value   : "
    f"€{current_market_value:,.0f}"
)

print(
    f"LSTM predicted value   : "
    f"€{prediction_eur:,.0f}"
)

print(
    f"Absolute change        : "
    f"€{absolute_change:+,.0f}"
)

print(
    f"Percentage change      : "
    f"{percentage_change:+.2f}%"
)

LSTM DEPLOYMENT PREDICTION
Player                 : Cole Palmer
Latest snapshot        : 2025-05-30
Approx. forecast date  : 2025-11-30
Current market value   : €120,000,000
LSTM predicted value   : €123,795,456
Absolute change        : €+3,795,456
Percentage change      : +3.16%


In [14]:
# ============================================================
# DEPLOYMENT PREDICTION SANITY CHECK
# ============================================================

prediction_is_finite = (
    np.isfinite(prediction_eur)
)

prediction_is_non_negative = (
    prediction_eur >= 0
)

prediction_is_valid = (
    prediction_is_finite
    and prediction_is_non_negative
)

print("DEPLOYMENT PREDICTION CHECK")
print("=" * 70)

print(
    f"Finite prediction      : "
    f"{prediction_is_finite}"
)

print(
    f"Non-negative prediction: "
    f"{prediction_is_non_negative}"
)

print(
    f"Valid prediction       : "
    f"{prediction_is_valid}"
)

print(
    f"\nFinal prediction       : "
    f"€{prediction_eur:,.0f}"
)

DEPLOYMENT PREDICTION CHECK
Finite prediction      : True
Non-negative prediction: True
Valid prediction       : True

Final prediction       : €123,795,456


In [15]:
# ============================================================
# REUSABLE LSTM DEPLOYMENT FUNCTION
# ============================================================

def predict_player_future_value(player_id):
    """
    Predict a player's future market value approximately
    six months after their latest available snapshot.

    The function:
    1. Finds the player's historical observations.
    2. Selects the latest three snapshots.
    3. Checks the 240-day maximum gap constraint.
    4. Checks required features.
    5. Scales the 3 x 4 sequence.
    6. Generates the LSTM prediction.
    7. Converts the prediction back to EUR.
    """

    # --------------------------------------------------------
    # 1. Find player history
    # --------------------------------------------------------

    history = (
        deployment_df[
            deployment_df["player_id"] == player_id
        ]
        .sort_values("snapshot_date")
        .reset_index(drop=True)
    )

    if history.empty:
        return {
            "success": False,
            "reason": "Player not found in temporal dataset."
        }

    # --------------------------------------------------------
    # 2. Check number of snapshots
    # --------------------------------------------------------

    if len(history) < sequence_length:
        return {
            "success": False,
            "reason": (
                f"Player has only {len(history)} snapshot(s). "
                f"At least {sequence_length} are required."
            )
        }

    # --------------------------------------------------------
    # 3. Select latest sequence
    # --------------------------------------------------------

    sequence = (
        history
        .tail(sequence_length)
        .copy()
        .reset_index(drop=True)
    )

    dates = sequence["snapshot_date"]

    gaps = (
        dates
        .diff()
        .dt.days
        .dropna()
        .to_numpy()
    )

    # --------------------------------------------------------
    # 4. Validate gaps
    # --------------------------------------------------------

    if (gaps > max_gap_days).any():
        return {
            "success": False,
            "reason": (
                "Latest historical sequence exceeds the "
                f"{max_gap_days}-day maximum gap constraint."
            ),
            "gaps_days": gaps.tolist()
        }

    # --------------------------------------------------------
    # 5. Validate model features
    # --------------------------------------------------------

    if sequence[model_features].isna().any().any():
        return {
            "success": False,
            "reason": (
                "Latest historical sequence contains "
                "missing model features."
            )
        }

    # --------------------------------------------------------
    # 6. Prepare raw sequence
    # --------------------------------------------------------

    sequence_raw = (
        sequence[model_features]
        .to_numpy(dtype=np.float64)
    )

    # --------------------------------------------------------
    # 7. Scale features
    # --------------------------------------------------------

    sequence_scaled = (
        feature_scaler
        .transform(sequence_raw)
        .reshape(
            1,
            sequence_length,
            len(model_features)
        )
    )

    # --------------------------------------------------------
    # 8. LSTM prediction
    # --------------------------------------------------------

    prediction_scaled = (
        lstm_model
        .predict(
            sequence_scaled,
            verbose=0
        )
        .reshape(-1, 1)
    )

    # --------------------------------------------------------
    # 9. Convert prediction back to EUR
    # --------------------------------------------------------

    prediction_eur = (
        target_scaler
        .inverse_transform(
            prediction_scaled
        )
        .reshape(-1)[0]
    )

    prediction_eur = max(
        0.0,
        float(prediction_eur)
    )

    # --------------------------------------------------------
    # 10. Current value and forecast change
    # --------------------------------------------------------

    current_value = float(
        sequence.iloc[-1][
            "current_market_value"
        ]
    )

    absolute_change = (
        prediction_eur
        - current_value
    )

    if current_value > 0:
        percentage_change = (
            absolute_change
            / current_value
            * 100
        )
    else:
        percentage_change = np.nan

    latest_snapshot = (
        sequence.iloc[-1][
            "snapshot_date"
        ]
    )

    forecast_date = (
        latest_snapshot
        + pd.DateOffset(months=6)
    )

    # --------------------------------------------------------
    # 11. Return deployment result
    # --------------------------------------------------------

    return {
        "success": True,

        "player_id": int(player_id),

        "latest_snapshot": latest_snapshot,

        "forecast_date": forecast_date,

        "current_market_value": current_value,

        "predicted_market_value": prediction_eur,

        "absolute_change": absolute_change,

        "percentage_change": percentage_change,

        "gaps_days": gaps.tolist(),

        "sequence": sequence[
            ["snapshot_date"] + model_features
        ].copy()
    }


print("LSTM deployment function created successfully.")

LSTM deployment function created successfully.


In [16]:
# ============================================================
# TEST REUSABLE FUNCTION: COLE PALMER
# ============================================================

cole_palmer_id = 568177

cole_result = predict_player_future_value(
    cole_palmer_id
)

print("REUSABLE DEPLOYMENT FUNCTION TEST")
print("=" * 70)

if cole_result["success"]:

    print("Prediction status       : SUCCESS")

    print(
        f"Player ID               : "
        f"{cole_result['player_id']}"
    )

    print(
        f"Latest snapshot         : "
        f"{cole_result['latest_snapshot'].date()}"
    )

    print(
        f"Forecast date           : "
        f"{cole_result['forecast_date'].date()}"
    )

    print(
        f"Current market value    : "
        f"€{cole_result['current_market_value']:,.0f}"
    )

    print(
        f"Predicted market value  : "
        f"€{cole_result['predicted_market_value']:,.0f}"
    )

    print(
        f"Absolute change         : "
        f"€{cole_result['absolute_change']:+,.0f}"
    )

    print(
        f"Percentage change       : "
        f"{cole_result['percentage_change']:+.2f}%"
    )

    print(
        f"Sequence gaps           : "
        f"{cole_result['gaps_days']} days"
    )

else:

    print("Prediction status : FAILED")

    print(
        f"Reason            : "
        f"{cole_result['reason']}"
    )

REUSABLE DEPLOYMENT FUNCTION TEST
Prediction status       : SUCCESS
Player ID               : 568177
Latest snapshot         : 2025-05-30
Forecast date           : 2025-11-30
Current market value    : €120,000,000
Predicted market value  : €123,795,456
Absolute change         : €+3,795,456
Percentage change       : +3.16%
Sequence gaps           : [203.0, 165.0] days


In [17]:
# ============================================================
# MANUAL VS FUNCTION CONSISTENCY CHECK
# ============================================================

function_prediction = (
    cole_result["predicted_market_value"]
)

predictions_match = np.isclose(
    prediction_eur,
    function_prediction,
    rtol=1e-7,
    atol=1e-2
)

print("DEPLOYMENT CONSISTENCY CHECK")
print("=" * 70)

print(
    f"Manual prediction   : "
    f"€{prediction_eur:,.2f}"
)

print(
    f"Function prediction : "
    f"€{function_prediction:,.2f}"
)

print(
    f"Predictions match   : "
    f"{predictions_match}"
)

DEPLOYMENT CONSISTENCY CHECK
Manual prediction   : €123,795,456.00
Function prediction : €123,795,456.00
Predictions match   : True


In [18]:
# ============================================================
# TEST INELIGIBLE PLAYER HANDLING
# ============================================================

snapshot_counts = (
    deployment_df
    .groupby("player_id")
    .size()
)

ineligible_candidates = (
    snapshot_counts[
        snapshot_counts < sequence_length
    ]
)

if len(ineligible_candidates) > 0:

    ineligible_player_id = int(
        ineligible_candidates.index[0]
    )

    ineligible_result = (
        predict_player_future_value(
            ineligible_player_id
        )
    )

    print("INELIGIBLE PLAYER TEST")
    print("=" * 70)

    print(
        f"Player ID : "
        f"{ineligible_player_id}"
    )

    print(
        f"Success   : "
        f"{ineligible_result['success']}"
    )

    print(
        f"Reason    : "
        f"{ineligible_result['reason']}"
    )

else:

    print(
        "No player with fewer than "
        f"{sequence_length} snapshots was found."
    )

INELIGIBLE PLAYER TEST
Player ID : 77
Success   : False
Reason    : Player has only 1 snapshot(s). At least 3 are required.


## Deployment Verification Conclusion

The deployment verification confirms that the saved LSTM model and preprocessing objects can be independently loaded and used to generate future market value predictions from historical player data.

The deployment pipeline constructs a sequence from the latest three valid player snapshots using four features: current market value, age, minutes played in the previous six months, and goals scored in the previous six months. Consecutive observations are required to satisfy the maximum 240-day gap constraint.

For the deployment example using Cole Palmer, the latest three observations occurred on 27 May 2024, 16 December 2024, and 30 May 2025. The saved LSTM model predicted a future market value of approximately €123.80 million from a current market value of €120 million, corresponding to an estimated increase of approximately 3.16%.

The reusable deployment function produced exactly the same prediction as the manually executed deployment pipeline, confirming implementation consistency. The function also correctly rejected a player with insufficient historical observations rather than generating a prediction from an invalid sequence.

Overall, the deployment test demonstrates that the final LSTM model can be reproducibly integrated into the application while enforcing the same sequence and temporal constraints used during model development.

In [19]:
# ============================================================
# LSTM DEPLOYMENT OBJECT CHECK
# ============================================================

objects_to_check = [
    "lstm_model",
    "lstm_raw_model",
    "feature_scaler",
    "target_scaler",
]

print("LSTM DEPLOYMENT OBJECT CHECK")
print("=" * 60)

for name in objects_to_check:
    if name in globals():
        obj = globals()[name]

        print(f"\n{name}")
        print(f"Status : FOUND")
        print(f"Type   : {type(obj)}")

    else:
        print(f"\n{name}")
        print("Status : NOT FOUND")


print("\n" + "=" * 60)
print("DEPLOYMENT FUNCTION")

if "predict_player_future_value" in globals():
    print("predict_player_future_value : FOUND")
else:
    print("predict_player_future_value : NOT FOUND")

LSTM DEPLOYMENT OBJECT CHECK

lstm_model
Status : FOUND
Type   : <class 'keras.src.models.sequential.Sequential'>

lstm_raw_model
Status : NOT FOUND

feature_scaler
Status : FOUND
Type   : <class 'sklearn.preprocessing._data.StandardScaler'>

target_scaler
Status : FOUND
Type   : <class 'sklearn.preprocessing._data.StandardScaler'>

DEPLOYMENT FUNCTION
predict_player_future_value : FOUND


In [20]:
# ============================================================
# CHECK SAVED LSTM DEPLOYMENT ARTIFACTS
# ============================================================

from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd()

# If notebook is running inside notebooks/...,
# move upward until project root is found
while (
    PROJECT_ROOT.name != "football-prediction"
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "experiment_2"
)

print("LSTM DEPLOYMENT ARTIFACT CHECK")
print("=" * 70)

print(f"Project root : {PROJECT_ROOT}")
print(f"Model folder : {MODEL_DIR}")
print()

if MODEL_DIR.exists():

    files = sorted(MODEL_DIR.iterdir())

    print("Files currently in models/experiment_2:")
    print("-" * 70)

    for file in files:
        print(file.name)

else:
    print("Model directory does not exist.")

LSTM DEPLOYMENT ARTIFACT CHECK
Project root : c:\Users\ngtht\football-prediction
Model folder : c:\Users\ngtht\football-prediction\models\experiment_2

Files currently in models/experiment_2:
----------------------------------------------------------------------
gradient_boosting_top4_future_value.joblib
gradient_boosting_top4_future_value_metadata.json
lstm_feature_scaler.joblib
lstm_future_market_value.keras
lstm_future_market_value_metadata.json
lstm_target_scaler.joblib


In [21]:
# ============================================================
# INDEPENDENT SAVED LSTM ARTIFACT TEST
# ============================================================

import joblib
import numpy as np
from tensorflow import keras

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

saved_lstm_path = (
    MODEL_DIR
    / "lstm_future_market_value.keras"
)

saved_feature_scaler_path = (
    MODEL_DIR
    / "lstm_feature_scaler.joblib"
)

saved_target_scaler_path = (
    MODEL_DIR
    / "lstm_target_scaler.joblib"
)

# ------------------------------------------------------------
# 2. Load fresh objects directly from disk
# ------------------------------------------------------------

saved_lstm_model = keras.models.load_model(
    saved_lstm_path
)

saved_feature_scaler = joblib.load(
    saved_feature_scaler_path
)

saved_target_scaler = joblib.load(
    saved_target_scaler_path
)

print("SAVED LSTM ARTIFACT LOAD TEST")
print("=" * 70)

print(
    "LSTM model loaded       :",
    saved_lstm_model is not None
)

print(
    "Feature scaler loaded   :",
    saved_feature_scaler is not None
)

print(
    "Target scaler loaded    :",
    saved_target_scaler is not None
)

# ------------------------------------------------------------
# 3. Use the same Cole Palmer raw sequence
# ------------------------------------------------------------

cole_history = (
    deployment_df[
        deployment_df["player_id"] == 568177
    ]
    .sort_values("snapshot_date")
    .tail(3)
    .copy()
)

cole_raw = (
    cole_history[model_features]
    .to_numpy(dtype=np.float64)
)

# ------------------------------------------------------------
# 4. Scale using SAVED scaler
# ------------------------------------------------------------

cole_scaled = (
    saved_feature_scaler
    .transform(cole_raw)
    .reshape(
        1,
        3,
        len(model_features)
    )
)

# ------------------------------------------------------------
# 5. Predict using SAVED model
# ------------------------------------------------------------

saved_prediction_scaled = (
    saved_lstm_model
    .predict(
        cole_scaled,
        verbose=0
    )
    .reshape(-1, 1)
)

# ------------------------------------------------------------
# 6. Convert prediction back to EUR
# ------------------------------------------------------------

saved_prediction_eur = (
    saved_target_scaler
    .inverse_transform(
        saved_prediction_scaled
    )
    .reshape(-1)[0]
)

saved_prediction_eur = max(
    0.0,
    float(saved_prediction_eur)
)

# ------------------------------------------------------------
# 7. Compare against verified deployment prediction
# ------------------------------------------------------------

same_prediction = np.isclose(
    saved_prediction_eur,
    cole_result["predicted_market_value"],
    rtol=1e-7,
    atol=1e-2
)

print()
print("SAVED MODEL PREDICTION")
print("=" * 70)

print(
    f"Verified function prediction : "
    f"€{cole_result['predicted_market_value']:,.2f}"
)

print(
    f"Fresh saved-file prediction  : "
    f"€{saved_prediction_eur:,.2f}"
)

print(
    f"Predictions match            : "
    f"{same_prediction}"
)

SAVED LSTM ARTIFACT LOAD TEST
LSTM model loaded       : True
Feature scaler loaded   : True
Target scaler loaded    : True

SAVED MODEL PREDICTION
Verified function prediction : €123,795,456.00
Fresh saved-file prediction  : €123,795,456.00
Predictions match            : True


In [24]:
# ============================================================
# REAL-WORLD LSTM FORECAST COMPARISON
# 15 PLAYER CASE STUDIES
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. CONFIGURATION
# ============================================================

N_PLAYERS = 15
SEQUENCE_LENGTH = 3
MAX_GAP_DAYS = 240
MAX_VALUATION_DISTANCE_DAYS = 30

MODEL_FEATURES = [
    "current_market_value",
    "age_at_snapshot",
    "minutes_last_6m",
    "goals_last_6m",
]


# ============================================================
# 2. PREPARE TEMPORAL DATA
# ============================================================

temporal_work = temporal_df.copy()

temporal_work["snapshot_date"] = pd.to_datetime(
    temporal_work["snapshot_date"],
    errors="coerce"
)

temporal_work["player_id"] = pd.to_numeric(
    temporal_work["player_id"],
    errors="coerce"
)

for feature in MODEL_FEATURES:
    temporal_work[feature] = pd.to_numeric(
        temporal_work[feature],
        errors="coerce"
    )

temporal_work = (
    temporal_work
    .dropna(
        subset=[
            "player_id",
            "snapshot_date"
        ]
    )
    .sort_values(
        [
            "player_id",
            "snapshot_date"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 3. PREPARE PLAYER NAMES
# ============================================================

players_work = players_df.copy()

players_work["player_id"] = pd.to_numeric(
    players_work["player_id"],
    errors="coerce"
)

# Find usable player-name column
possible_name_columns = [
    "name",
    "pretty_name",
    "player_name"
]

name_column = None

for col in possible_name_columns:
    if col in players_work.columns:
        name_column = col
        break

if name_column is None:
    raise ValueError(
        "Could not find a player-name column in players_df."
    )

player_name_lookup = (
    players_work[
        [
            "player_id",
            name_column
        ]
    ]
    .dropna(subset=["player_id"])
    .drop_duplicates("player_id")
    .rename(
        columns={
            name_column: "Player"
        }
    )
)


# ============================================================
# 4. LOAD HISTORICAL MARKET VALUATIONS
# ============================================================

VALUATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "player_valuations.csv"
)

valuation_lookup = pd.read_csv(
    VALUATIONS_PATH,
    low_memory=False
)

valuation_lookup["player_id"] = pd.to_numeric(
    valuation_lookup["player_id"],
    errors="coerce"
)

valuation_lookup["date"] = pd.to_datetime(
    valuation_lookup["date"],
    errors="coerce"
)

valuation_lookup["market_value_in_eur"] = pd.to_numeric(
    valuation_lookup["market_value_in_eur"],
    errors="coerce"
)

valuation_lookup = (
    valuation_lookup
    .dropna(
        subset=[
            "player_id",
            "date",
            "market_value_in_eur"
        ]
    )
    .sort_values(
        [
            "player_id",
            "date"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Valuation date range:",
    valuation_lookup["date"].min().date(),
    "→",
    valuation_lookup["date"].max().date()
)


# ============================================================
# 5. LSTM SEQUENCE CHECK
# ============================================================

def build_lstm_sequence(player_id):

    history = (
        temporal_work[
            temporal_work["player_id"]
            == player_id
        ]
        .sort_values("snapshot_date")
        .reset_index(drop=True)
    )

    # Need at least 3 observations
    if len(history) < SEQUENCE_LENGTH:
        return None

    # Latest 3 observations
    sequence = (
        history
        .tail(SEQUENCE_LENGTH)
        .copy()
        .reset_index(drop=True)
    )

    # All four features must exist
    if (
        sequence[MODEL_FEATURES]
        .isna()
        .any()
        .any()
    ):
        return None

    # Calculate gaps
    gaps = (
        sequence["snapshot_date"]
        .diff()
        .dt.days
        .dropna()
        .to_numpy()
    )

    if len(gaps) != SEQUENCE_LENGTH - 1:
        return None

    # Same maximum-gap rule as LSTM deployment
    if (
        (gaps <= 0).any()
        or
        (gaps > MAX_GAP_DAYS).any()
    ):
        return None

    return sequence


# ============================================================
# 6. LSTM PREDICTION
# ============================================================

def generate_lstm_prediction(sequence):

    raw_sequence = (
        sequence[MODEL_FEATURES]
        .to_numpy(dtype=np.float64)
    )

    if raw_sequence.shape != (3, 4):
        return None

    if not np.isfinite(raw_sequence).all():
        return None

    # Scale each observation
    scaled_2d = feature_scaler.transform(
        raw_sequence
    )

    # LSTM input shape:
    # batch x timesteps x features
    # 1 x 3 x 4
    scaled_3d = scaled_2d.reshape(
        1,
        SEQUENCE_LENGTH,
        len(MODEL_FEATURES)
    )

    prediction_scaled = (
        lstm_model
        .predict(
            scaled_3d,
            verbose=0
        )
        .reshape(-1, 1)
    )

    prediction_eur = (
        target_scaler
        .inverse_transform(
            prediction_scaled
        )
        .reshape(-1)[0]
    )

    if not np.isfinite(prediction_eur):
        return None

    return max(
        0.0,
        float(prediction_eur)
    )


# ============================================================
# 7. FIND EXTERNAL VALUE CLOSEST TO FORECAST DATE
# ============================================================

def find_closest_valuation(
    player_id,
    forecast_date
):

    values = (
        valuation_lookup[
            valuation_lookup["player_id"]
            == player_id
        ]
        .copy()
    )

    if values.empty:
        return None

    values["Date Gap (Days)"] = (
        values["date"]
        - forecast_date
    ).dt.days

    values["Absolute Gap"] = (
        values["Date Gap (Days)"]
        .abs()
    )

    closest = (
        values
        .sort_values(
            [
                "Absolute Gap",
                "date"
            ]
        )
        .iloc[0]
    )

    # Reject valuations too far from forecast target
    if (
        closest["Absolute Gap"]
        > MAX_VALUATION_DISTANCE_DAYS
    ):
        return None

    return {
        "value":
            float(
                closest["market_value_in_eur"]
            ),

        "date":
            pd.Timestamp(
                closest["date"]
            ),

        "gap":
            int(
                closest["Date Gap (Days)"]
            )
    }


# ============================================================
# 8. GET LATEST SNAPSHOT FOR EACH PLAYER
# ============================================================

latest_players = (
    temporal_work
    .groupby(
        "player_id",
        as_index=False
    )
    .tail(1)
    .copy()
)

latest_players = (
    latest_players
    .merge(
        player_name_lookup,
        on="player_id",
        how="left"
    )
)

# Prioritise recognisable / higher-value players
latest_players = (
    latest_players
    .sort_values(
        "current_market_value",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 9. GENERATE COMPARISON
# ============================================================

results = []


for _, latest_row in latest_players.iterrows():

    player_id = int(
        latest_row["player_id"]
    )

    player_name = latest_row["Player"]

    if pd.isna(player_name):
        continue

    # ----------------------------------------
    # Build valid 3-observation sequence
    # ----------------------------------------

    sequence = build_lstm_sequence(
        player_id
    )

    if sequence is None:
        continue


    # ----------------------------------------
    # LSTM prediction
    # ----------------------------------------

    prediction = generate_lstm_prediction(
        sequence
    )

    if prediction is None:
        continue


    # ----------------------------------------
    # Forecast date
    # ----------------------------------------

    snapshot_date = pd.Timestamp(
        sequence.iloc[-1]["snapshot_date"]
    )

    forecast_date = (
        snapshot_date
        + pd.DateOffset(months=6)
    )


    # ----------------------------------------
    # External valuation
    # ----------------------------------------

    external = find_closest_valuation(
        player_id,
        forecast_date
    )

    if external is None:
        continue


    external_value = external["value"]

    difference_eur = (
        prediction
        - external_value
    )

    difference_pct = (
        difference_eur
        / external_value
        * 100
    )


    results.append(
        {
            "Player":
                str(player_name),

            "Player ID":
                player_id,

            "Snapshot Date":
                snapshot_date,

            "Forecast Date":
                forecast_date,

            "LSTM Forecast":
                prediction,

            "External Market Value":
                external_value,

            "Valuation Date":
                external["date"],

            "Date Gap (Days)":
                external["gap"],

            "Difference (€)":
                difference_eur,

            "Difference (%)":
                difference_pct
        }
    )


    if len(results) >= N_PLAYERS:
        break


# ============================================================
# 10. CREATE DATAFRAME
# ============================================================

comparison_df = pd.DataFrame(results)

print(
    "\nValid comparisons found:",
    len(comparison_df)
)


# ============================================================
# 11. FULL RESEARCH TABLE
# ============================================================

if comparison_df.empty:

    print(
        "No valid comparisons found."
    )

else:

    research_table = comparison_df.copy()

    for column in [
        "Snapshot Date",
        "Forecast Date",
        "Valuation Date"
    ]:

        research_table[column] = (
            pd.to_datetime(
                research_table[column]
            )
            .dt.strftime("%d %b %Y")
        )


    research_table["LSTM Forecast"] = (
        research_table["LSTM Forecast"]
        .map(
            lambda x:
            f"€{x / 1_000_000:.2f}M"
        )
    )

    research_table[
        "External Market Value"
    ] = (
        research_table[
            "External Market Value"
        ]
        .map(
            lambda x:
            f"€{x / 1_000_000:.2f}M"
        )
    )

    research_table["Difference (€)"] = (
        research_table["Difference (€)"]
        .map(
            lambda x:
            f"€{x / 1_000_000:+.2f}M"
        )
    )

    research_table["Difference (%)"] = (
        research_table["Difference (%)"]
        .map(
            lambda x:
            f"{x:+.2f}%"
        )
    )

    display(research_table)


# ============================================================
# 12. POWERPOINT TABLE
# ============================================================

if not comparison_df.empty:

    slide_table = comparison_df[
        [
            "Player",
            "LSTM Forecast",
            "External Market Value",
            "Difference (€)",
            "Difference (%)"
        ]
    ].copy()


    slide_table["LSTM Forecast"] = (
        slide_table["LSTM Forecast"]
        .map(
            lambda x:
            f"€{x / 1_000_000:.2f}M"
        )
    )


    slide_table[
        "External Market Value"
    ] = (
        slide_table[
            "External Market Value"
        ]
        .map(
            lambda x:
            f"€{x / 1_000_000:.2f}M"
        )
    )


    slide_table["Difference (€)"] = (
        slide_table["Difference (€)"]
        .map(
            lambda x:
            f"€{x / 1_000_000:+.2f}M"
        )
    )


    slide_table["Difference (%)"] = (
        slide_table["Difference (%)"]
        .map(
            lambda x:
            f"{x:+.2f}%"
        )
    )


    print(
        "\nPOWERPOINT TABLE"
    )

    display(slide_table)


# ============================================================
# 13. SUMMARY
# ============================================================

if not comparison_df.empty:

    mad = (
        comparison_df["Difference (€)"]
        .abs()
        .mean()
    )

    mapd = (
        comparison_df["Difference (%)"]
        .abs()
        .mean()
    )

    within_10 = (
        comparison_df["Difference (%)"]
        .abs()
        .le(10)
        .sum()
    )

    over = (
        comparison_df["Difference (%)"]
        .gt(0)
        .sum()
    )

    under = (
        comparison_df["Difference (%)"]
        .lt(0)
        .sum()
    )


    print("\n==============================")
    print("SUMMARY")
    print("==============================")

    print(
        "Mean Absolute Difference:",
        f"€{mad / 1_000_000:.2f}M"
    )

    print(
        "Mean Absolute Percentage Difference:",
        f"{mapd:.2f}%"
    )

    print(
        "Within ±10%:",
        f"{within_10}/{len(comparison_df)}"
    )

    print(
        "Over-predictions:",
        over
    )

    print(
        "Under-predictions:",
        under
    )

Valuation date range: 2000-01-20 → 2026-05-18

Valid comparisons found: 15


,Player,Player ID,Snapshot Date,Forecast Date,LSTM Forecast,External Market Value,Valuation Date,Date Gap (Days),Difference (€),Difference (%)
0,Lamine Yamal,937958,09 Jun 2025,09 Dec 2025,€185.58M,€200.00M,12 Dec 2025,3,€-14.42M,-7.21%
1,Erling Haaland,418560,30 May 2025,30 Nov 2025,€172.47M,€200.00M,09 Dec 2025,9,€-27.53M,-13.76%
2,Kylian Mbappé,342229,09 Jun 2025,09 Dec 2025,€175.25M,€200.00M,12 Dec 2025,3,€-24.75M,-12.37%
3,Jude Bellingham,581678,09 Jun 2025,09 Dec 2025,€174.53M,€160.00M,12 Dec 2025,3,€+14.53M,+9.08%
4,Vinicius Junior,371998,09 Jun 2025,09 Dec 2025,€163.69M,€150.00M,12 Dec 2025,3,€+13.69M,+9.13%
5,Jamal Musiala,580195,05 Jun 2025,05 Dec 2025,€142.33M,€130.00M,19 Dec 2025,14,€+12.33M,+9.49%
6,Pedri,683840,09 Jun 2025,09 Dec 2025,€142.89M,€140.00M,12 Dec 2025,3,€+2.89M,+2.07%
7,Florian Wirtz,598577,05 Jun 2025,05 Dec 2025,€142.84M,€110.00M,09 Dec 2025,4,€+32.84M,+29.86%
8,Federico Valverde,369081,09 Jun 2025,09 Dec 2025,€130.76M,€120.00M,12 Dec 2025,3,€+10.76M,+8.96%
9,Michael Olise,566723,14 Oct 2025,14 Apr 2026,€137.77M,€140.00M,20 Mar 2026,-25,€-2.23M,-1.59%



POWERPOINT TABLE


,Player,LSTM Forecast,External Market Value,Difference (€),Difference (%)
0,Lamine Yamal,€185.58M,€200.00M,€-14.42M,-7.21%
1,Erling Haaland,€172.47M,€200.00M,€-27.53M,-13.76%
2,Kylian Mbappé,€175.25M,€200.00M,€-24.75M,-12.37%
3,Jude Bellingham,€174.53M,€160.00M,€+14.53M,+9.08%
4,Vinicius Junior,€163.69M,€150.00M,€+13.69M,+9.13%
5,Jamal Musiala,€142.33M,€130.00M,€+12.33M,+9.49%
6,Pedri,€142.89M,€140.00M,€+2.89M,+2.07%
7,Florian Wirtz,€142.84M,€110.00M,€+32.84M,+29.86%
8,Federico Valverde,€130.76M,€120.00M,€+10.76M,+8.96%
9,Michael Olise,€137.77M,€140.00M,€-2.23M,-1.59%



SUMMARY
Mean Absolute Difference: €13.92M
Mean Absolute Percentage Difference: 10.82%
Within ±10%: 10/15
Over-predictions: 11
Under-predictions: 4
